# Python Blockchain From Scratch

This notebook demonstrates the consolidated implementation in `blockchain.py`: wallets, signed transactions, SHA-256 block hashes, Proof of Work, balances, and tamper detection.

> Educational prototype only — not production cryptocurrency software.


In [7]:
from blockchain import Blockchain, Transaction, Wallet


## 1. Create wallets and initialize demo balances

Each wallet creates an RSA key pair. Its address is the SHA-256 digest of its public key. Starting balances are assigned directly for this demonstration.


In [8]:
alice = Wallet("Alice")
bob = Wallet("Bob")
blockchain = Blockchain(difficulty=4)

blockchain.set_balance(alice, 1_000)
blockchain.set_balance(bob, 100)

print("Alice address:", alice.address)
print("Bob address:  ", bob.address)
print("Initial balances:", blockchain.balances)


Alice address: 593a90bdfe42b4242180a737e05925f0ca2bdc16d75cd2c8b9d437c354969e3c
Bob address:   b0b81c9391a544030f540b409f2aa20476dc2a0ee1287bee3abaf972ca4f5f12
Initial balances: {'593a90bdfe42b4242180a737e05925f0ca2bdc16d75cd2c8b9d437c354969e3c': 1000.0, 'b0b81c9391a544030f540b409f2aa20476dc2a0ee1287bee3abaf972ca4f5f12': 100.0}


## 2. Create and sign a transaction

Alice signs the transaction using her private key. The blockchain verifies the signature before accepting it into the pending transaction pool.


In [9]:
transaction = Transaction(alice, bob, 150)
transaction.sign()

print("Signature valid:", transaction.verify())
print("Transaction accepted:", blockchain.add_transaction(transaction))
print("Pending transactions:", len(blockchain.pending_transactions))


Signature valid: True
Transaction accepted: True
Pending transactions: 1


## 3. Process the transaction and mine a block

The miner searches for a nonce that makes the block hash begin with four zeroes.


In [10]:
blockchain.process_pending_transactions()

print("Alice balance:", blockchain.balances[alice.address])
print("Bob balance:  ", blockchain.balances[bob.address])
print("Number of blocks:", len(blockchain.chain))
print("Blockchain valid:", blockchain.is_valid())


Alice balance: 850.0
Bob balance:   250.0
Number of blocks: 2
Blockchain valid: True


In [11]:
mined_block = blockchain.chain[-1]
print("Previous hash:", mined_block.previous_hash)
print("Current hash: ", mined_block.current_hash)
print("Nonce:        ", mined_block.nonce)
print("Stored transaction:", mined_block.transactions[0])


Previous hash: 357d84906d3bb95409016c5f68449d4e247b4a9b509b32492277398d809c6596
Current hash:  00008cf318d2e1e42f800c4718c30e2fd637e29e6c221a9c70c43427d0bb61a5
Nonce:         6974
Stored transaction: {'sender': '-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAsf55GAsweokIYVbMF9RQ\nQqJvh2rIckShnpQjc9+P175t7vNghzBBLR6VaHsh8OSPizOhmaX/GFwsSI2raOht\n35nJIeIqthhKUbeZ4jkYbrfSEqbEXjX+cRDrcPCQVEuLLtA4m7TCgXc98c9nB5qE\n8VjBd0vxAB7LeE999KvhWZGRH5dumat2D44NYtEZtjDVU2/w14UKyc/OCh6pbONV\nAkcNS21y9QU4F6gZJWsmpY5t35lEinSOtkhKR2ZR5v+12IWvXKfbip7L/cG0GX+W\nlTaWLEKna8w5/XBBZ1FECBTIeJKhzoshMLNHpkVAtc5mdSkNP+vGCCwb+mYJzaV/\n5wIDAQAB\n-----END PUBLIC KEY-----\n', 'receiver': '-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAsEV+6r2bgTbYjzpA95iM\ni1ZXkb4gy4yyj+HfhDwYKMtyqY5iLNyHQgkoh0ieSY6sTFOAKMuTwh3wuxJTsxgP\n3lvKsPVVlMfEw6QLyfJGUo3rXn3TNAOKGSK8XUMaQyGwaarW873SgrOWHOabad3K\nBbQoefZd0+sO2pRDmCFCHNQBAViV6AvHYIYM38Cp8lxuYnOM2JSilTIzVBupMil5\nuYb/UPghR0mMT1mKVjxx

## 4. Demonstrate tamper detection

Change the amount stored in the mined block. The block hash and its digital signature will no longer validate.


In [12]:
mined_block.transactions[0]["amount"] = 999
print("Blockchain valid after tampering:", blockchain.is_valid())


Blockchain valid after tampering: False
